In [3]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
from pathlib import Path
from IPython.display import IFrame

In [4]:
OUT_DIR = Path("kg_output")
nodes_df = pd.read_csv(OUT_DIR / "kg_nodes.csv")
edges_df = pd.read_csv(OUT_DIR / "kg_edges.csv")
 
print(f"Nodes: {len(nodes_df):,}")
print(f"Edges: {len(edges_df):,}")
print(nodes_df["label"].value_counts())

# 加载CSV

Nodes: 3,460
Edges: 23,051
label
FundusImage     689
OpticDisc       689
NeuralRim       689
Pathology       689
Diagnosis       689
ClinicalRule     11
RiskLevel         4
Name: count, dtype: int64


In [5]:
NODE_STYLE = {
    "FundusImage":  {"color": "#7B68EE", "shape": "image",  "size": 28},
    "OpticDisc":    {"color": "#20B2AA", "shape": "dot",    "size": 22},
    "NeuralRim":    {"color": "#3CB371", "shape": "dot",    "size": 22},
    "Pathology":    {"color": "#FF8C00", "shape": "dot",    "size": 22},
    "Diagnosis":    {"color": "#DC143C", "shape": "diamond","size": 26},
    "RiskLevel":    {"color": "#B22222", "shape": "star",   "size": 24},
    "ClinicalRule": {"color": "#4169E1", "shape": "square", "size": 20},
}
 
EDGE_STYLE = {
    "HAS_OPTIC_DISC":    {"color": "#20B2AA", "width": 1.5, "dashes": False},
    "HAS_RIM":           {"color": "#3CB371", "width": 1.5, "dashes": False},
    "HAS_PATHOLOGY":     {"color": "#FF8C00", "width": 1.5, "dashes": False},
    "HAS_DIAGNOSIS":     {"color": "#DC143C", "width": 2.0, "dashes": False},
    "HAS_RISK":          {"color": "#B22222", "width": 1.5, "dashes": False},
    "SUPPORTS_DIAGNOSIS":{"color": "#999999", "width": 1.0, "dashes": True},
    "GOVERNED_BY":       {"color": "#4169E1", "width": 1.0, "dashes": True},
    "SUPPORTS_RULE":     {"color": "#6A5ACD", "width": 1.0, "dashes": True},
}

# 要好看看，美美哒。肯定要选好配色

In [6]:
def build_schema_graph():
    net = Network(height="600px", width="100%", bgcolor="#1a1a2e",
                  font_color="white", directed=True)
    net.set_options("""
    {
      "physics": {"enabled": true, "stabilization": {"iterations": 200}},
      "edges":   {"arrows": {"to": {"enabled": true, "scaleFactor": 0.8}}},
      "nodes":   {"font": {"size": 16, "face": "Arial"}}
    }
    """)
 
    # Schema 节点
    schema_nodes = list(NODE_STYLE.keys())
    for label in schema_nodes:
        s = NODE_STYLE[label]
        net.add_node(label, label=label, color=s["color"],
                     shape=s["shape"] if s["shape"] != "image" else "dot",
                     size=s["size"] + 5, title=label, font={"size": 16})
 
    # Schema 边（代表性关系）
    schema_edges = [
        ("FundusImage", "OpticDisc",    "HAS_OPTIC_DISC"),
        ("FundusImage", "NeuralRim",    "HAS_RIM"),
        ("FundusImage", "Pathology",    "HAS_PATHOLOGY"),
        ("FundusImage", "Diagnosis",    "HAS_DIAGNOSIS"),
        ("Diagnosis",   "RiskLevel",    "HAS_RISK"),
        ("OpticDisc",   "Diagnosis",    "SUPPORTS_DIAGNOSIS"),
        ("NeuralRim",   "Diagnosis",    "SUPPORTS_DIAGNOSIS"),
        ("Pathology",   "Diagnosis",    "SUPPORTS_DIAGNOSIS"),
        ("OpticDisc",   "ClinicalRule", "GOVERNED_BY"),
        ("NeuralRim",   "ClinicalRule", "GOVERNED_BY"),
        ("Pathology",   "ClinicalRule", "GOVERNED_BY"),
        ("Diagnosis",   "ClinicalRule", "SUPPORTS_RULE"),
    ]
    for src, dst, rel in schema_edges:
        s = EDGE_STYLE.get(rel, {"color": "#aaaaaa", "width": 1.5, "dashes": False})
        net.add_edge(src, dst, label=rel, color=s["color"],
                     width=s["width"], dashes=s["dashes"],
                     font={"size": 11, "color": "#cccccc"})
 
    path = OUT_DIR / "fig1_schema.html"
    net.save_graph(str(path))
    print(f"Saved: {path}")
    return path
 
schema_path = build_schema_graph()
IFrame(str(schema_path), width="100%", height=620)

# Cell 4 — 图1: Schema 图（论文 Figure 1）
# 只展示节点类型和关系类型，不展示具体数据
 

Saved: kg_output/fig1_schema.html


In [ ]:
def build_case_subgraph(annotation="glaucoma", risk="high risk", case_idx=0):
    # 找目标 FundusImage
    img_nodes = nodes_df[
        (nodes_df.label == "FundusImage") &
        (nodes_df.annotation == annotation)
    ]
    img_id = img_nodes.iloc[case_idx]["id"]
    filename = img_nodes.iloc[case_idx].get("filename", img_id)
 
    # BFS 收集该图像的所有关联节点（2跳）
    relevant_ids = {img_id}
    for _ in range(2):
        connected = edges_df[
            edges_df["src"].isin(relevant_ids) | edges_df["dst"].isin(relevant_ids)
        ]
        relevant_ids |= set(connected["src"]) | set(connected["dst"])
 
    sub_nodes = nodes_df[nodes_df["id"].isin(relevant_ids)]
    sub_edges = edges_df[
        edges_df["src"].isin(relevant_ids) & edges_df["dst"].isin(relevant_ids)
    ]
 
    net = Network(height="650px", width="100%", bgcolor="#1a1a2e",
                  font_color="white", directed=True)
    net.set_options("""
    {
      "physics": {
        "enabled": true,
        "barnesHut": {"gravitationalConstant": -8000, "springLength": 180},
        "stabilization": {"iterations": 300}
      },
      "edges": {"arrows": {"to": {"enabled": true, "scaleFactor": 0.8}}}
    }
    """)
 
    node_lookup = sub_nodes.set_index("id")
 
    for _, row in sub_nodes.iterrows():
        lbl   = row.get("label", "")
        s     = NODE_STYLE.get(lbl, {"color": "#888888", "shape": "dot", "size": 18})
 
        # 节点 tooltip
        props = {k: v for k, v in row.items()
                 if k not in ("id", "label") and pd.notna(v) and str(v) != "nan"}
        tooltip = f"<b>{lbl}</b><br>" + "<br>".join(
            f"{k}: {v}" for k, v in list(props.items())[:8]
        )
 
        # 显示标签
        if lbl == "FundusImage":
            display = str(filename).split("/")[-1]
        elif lbl == "ClinicalRule":
            display = str(row.get("name", row["id"])).replace(" ", "\n")
        elif lbl == "RiskLevel":
            display = str(row.get("value", row["id"]))
        elif lbl == "Diagnosis":
            display = f"Diagnosis\n{row.get('risk_assessment','')}"
        else:
            display = lbl
 
        net.add_node(
            row["id"], label=display,
            color=s["color"],
            shape=s["shape"] if s["shape"] != "image" else "dot",
            size=s["size"], title=tooltip,
            font={"size": 13}
        )
 
    for _, row in sub_edges.iterrows():
        rel = str(row.get("rel", ""))
        s   = EDGE_STYLE.get(rel, {"color": "#aaaaaa", "width": 1.2, "dashes": False})
        evidence = row.get("evidence", "")
        elabel   = rel if pd.isna(evidence) or str(evidence) == "nan" else f"{rel}\n({evidence})"
        net.add_edge(
            str(row["src"]), str(row["dst"]),
            label=elabel, color=s["color"],
            width=s["width"], dashes=s["dashes"],
            font={"size": 10, "color": "#bbbbbb"}
        )
 
    path = OUT_DIR / f"fig2_case_{annotation}_{case_idx}.html"
    net.save_graph(str(path))
    print(f"Saved: {path}  ({len(sub_nodes)} nodes, {len(sub_edges)} edges)")
    return path
 
case_path = build_case_subgraph("glaucoma", "high risk", case_idx=0)
IFrame(str(case_path), width="100%", height=670)

# Cell 5 — 图2: 单病例子图（论文 Figure 2）
# 取1个 glaucoma high-risk 病例，展示完整推理链

Saved: kg_output/fig2_case_glaucoma_0.html  (17 nodes, 38 edges)


In [ ]:
def build_overview_graph(n_glaucoma=15, n_normal=10):
    # 采样图像节点
    glaucoma_imgs = nodes_df[
        (nodes_df.label == "FundusImage") & (nodes_df.annotation == "glaucoma")
    ].sample(n=min(n_glaucoma, len(nodes_df[nodes_df.annotation=="glaucoma"])), random_state=42)["id"]
 
    normal_imgs = nodes_df[
        (nodes_df.label == "FundusImage") & (nodes_df.annotation == "normal")
    ].sample(n=min(n_normal, len(nodes_df[nodes_df.annotation=="normal"])), random_state=42)["id"]
 
    selected_imgs = set(glaucoma_imgs) | set(normal_imgs)
 
    # 收集1跳邻居
    relevant_ids = set(selected_imgs)
    connected = edges_df[edges_df["src"].isin(selected_imgs)]
    relevant_ids |= set(connected["dst"])
    # 加入所有 ClinicalRule 节点
    rule_ids = set(nodes_df[nodes_df.label == "ClinicalRule"]["id"])
    relevant_ids |= rule_ids
    # 加入 RiskLevel 节点
    risk_ids = set(nodes_df[nodes_df.label == "RiskLevel"]["id"])
    relevant_ids |= risk_ids
 
    sub_nodes = nodes_df[nodes_df["id"].isin(relevant_ids)]
    sub_edges = edges_df[
        edges_df["src"].isin(relevant_ids) & edges_df["dst"].isin(relevant_ids)
    ]
 
    net = Network(height="750px", width="100%", bgcolor="#0d0d1a",
                  font_color="white", directed=True)
    net.set_options("""
    {
      "physics": {
        "enabled": true,
        "barnesHut": {
          "gravitationalConstant": -5000,
          "centralGravity": 0.3,
          "springLength": 140
        },
        "stabilization": {"iterations": 400}
      },
      "edges": {
        "arrows": {"to": {"enabled": true, "scaleFactor": 0.6}},
        "smooth":  {"type": "continuous"}
      }
    }
    """)
 
    for _, row in sub_nodes.iterrows():
        lbl = row.get("label", "")
        s   = NODE_STYLE.get(lbl, {"color": "#888888", "shape": "dot", "size": 16})
        ann = str(row.get("annotation", ""))
 
        # glaucoma 图像节点加红色边框
        border_color = "#FF4444" if ann == "glaucoma" else (
                       "#44FF44" if ann == "normal" else s["color"])
 
        if lbl == "FundusImage":
            display = ann
        elif lbl == "ClinicalRule":
            display = str(row.get("name", "")).replace(" ", "\n")
        elif lbl == "RiskLevel":
            display = str(row.get("value", ""))
        elif lbl == "Diagnosis":
            display = str(row.get("risk_assessment", "diag"))
        else:
            display = lbl
 
        net.add_node(
            row["id"], label=display,
            color={"background": s["color"], "border": border_color,
                   "highlight": {"background": s["color"], "border": "#ffffff"}},
            shape=s["shape"] if s["shape"] != "image" else "dot",
            size=s["size"], title=lbl, font={"size": 11}
        )
 
    for _, row in sub_edges.iterrows():
        rel = str(row.get("rel", ""))
        s   = EDGE_STYLE.get(rel, {"color": "#555555", "width": 0.8, "dashes": False})
        net.add_edge(
            str(row["src"]), str(row["dst"]),
            color=s["color"], width=s["width"], dashes=s["dashes"],
            title=rel
        )
 
    path = OUT_DIR / "fig3_overview.html"
    net.save_graph(str(path))
    print(f"Saved: {path}  ({len(sub_nodes)} nodes, {len(sub_edges)} edges)")
    return path
 
overview_path = build_overview_graph(n_glaucoma=15, n_normal=10)
IFrame(str(overview_path), width="100%", height=770)

# Cell 6 — 图3: 全图概览（论文 Figure 3）
# 采样展示，避免节点过多

Saved: kg_output/fig3_overview.html  (140 nodes, 847 edges)


In [11]:
# Cell — 修复独立 HTML（去掉本地依赖，改用 CDN）
import re
from pathlib import Path

OUT_DIR = Path("kg_output")

for html_file in OUT_DIR.glob("fig*.html"):
    content = html_file.read_text(encoding="utf-8")
    # 替换本地 utils.js 引用为空（pyvis 不需要它也能运行）
    content = content.replace(
        '<script src="lib/bindings/utils.js"></script>', ''
    )
    html_file.write_text(content, encoding="utf-8")
    print(f"Fixed: {html_file.name}")

print("现在可以直接双击 HTML 文件在浏览器打开")

# 真的太丑了，我操

Fixed: fig3_overview.html
Fixed: fig2_case_glaucoma_0.html
Fixed: fig1_schema.html
现在可以直接双击 HTML 文件在浏览器打开


In [12]:
# ═══════════════════════════════════════════════════════════
# Cell — 图1 重做：固定坐标布局，论文级别
# ═══════════════════════════════════════════════════════════
from pyvis.network import Network
from pathlib import Path
from IPython.display import IFrame

OUT_DIR = Path("kg_output")

net = Network(height="680px", width="100%", bgcolor="#1a1a2e",
              font_color="white", directed=True)

# 关闭物理引擎，完全用固定坐标
net.set_options("""
{
  "physics": {"enabled": false},
  "edges": {
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.9}},
    "smooth": {"type": "curvedCW", "roundness": 0.15},
    "font":   {"size": 12, "color": "#dddddd", "strokeWidth": 2, "strokeColor": "#1a1a2e"}
  },
  "nodes": {
    "font": {"size": 15, "face": "Arial", "bold": true}
  },
  "interaction": {"dragNodes": true, "zoomView": true}
}
""")

# ── 固定坐标（六边形放射状布局）──────────────────────────
#
#          OpticDisc(top)
#   NeuralRim            ClinicalRule
#       FundusImage(center)
#   Pathology            RiskLevel
#          Diagnosis(bottom)
#
NODES = [
    ("FundusImage",  0,    0,   "#7B68EE", "dot",     32),
    ("OpticDisc",    0,   -240, "#20B2AA", "dot",     26),
    ("NeuralRim",   -220, -120, "#3CB371", "dot",     26),
    ("Pathology",   -220,  120, "#FF8C00", "dot",     26),
    ("Diagnosis",    0,    240, "#DC143C", "diamond", 30),
    ("RiskLevel",    220,  240, "#B22222", "star",    28),
    ("ClinicalRule", 220, -120, "#4169E1", "square",  24),
]

for nid, x, y, color, shape, size in NODES:
    net.add_node(
        nid, label=nid, color=color, shape=shape, size=size,
        x=x, y=y, physics=False,
        font={"size": 15, "color": "white", "bold": True},
        title=nid,
    )

# ── 边定义 ─────────────────────────────────────────────────
EDGES = [
    # 结构边（实线）
    ("FundusImage", "OpticDisc",    "HAS_OPTIC_DISC",     "#20B2AA", 2.0, False),
    ("FundusImage", "NeuralRim",    "HAS_RIM",            "#3CB371", 2.0, False),
    ("FundusImage", "Pathology",    "HAS_PATHOLOGY",      "#FF8C00", 2.0, False),
    ("FundusImage", "Diagnosis",    "HAS_DIAGNOSIS",      "#DC143C", 2.5, False),
    ("Diagnosis",   "RiskLevel",    "HAS_RISK",           "#B22222", 2.0, False),
    # 推理边（虚线）
    ("OpticDisc",   "Diagnosis",    "SUPPORTS_DIAGNOSIS", "#888888", 1.2, True),
    ("NeuralRim",   "Diagnosis",    "SUPPORTS_DIAGNOSIS", "#888888", 1.2, True),
    ("Pathology",   "Diagnosis",    "SUPPORTS_DIAGNOSIS", "#888888", 1.2, True),
    # 规则边（虚线）
    ("OpticDisc",   "ClinicalRule", "GOVERNED_BY",        "#4169E1", 1.2, True),
    ("NeuralRim",   "ClinicalRule", "GOVERNED_BY",        "#4169E1", 1.2, True),
    ("Pathology",   "ClinicalRule", "GOVERNED_BY",        "#4169E1", 1.2, True),
    ("Diagnosis",   "ClinicalRule", "SUPPORTS_RULE",      "#6A5ACD", 1.2, True),
]

for src, dst, label, color, width, dashes in EDGES:
    net.add_edge(
        src, dst, label=label,
        color=color, width=width, dashes=dashes,
        font={"size": 11, "color": "#cccccc",
              "strokeWidth": 2, "strokeColor": "#1a1a2e"},
        smooth={"type": "curvedCW", "roundness": 0.2},
    )

# 保存并修复本地依赖
path = OUT_DIR / "fig1_schema_v2.html"
net.save_graph(str(path))

# 修复 utils.js 本地引用
content = path.read_text(encoding="utf-8")
content = content.replace('<script src="lib/bindings/utils.js"></script>', '')
path.write_text(content, encoding="utf-8")

print(f"Saved: {path}")
IFrame(str(path), width="100%", height=700)

Saved: kg_output/fig1_schema_v2.html


In [13]:
# ═══════════════════════════════════════════════════════════
# Cell — 图2 重做：单病例推理链，固定分层布局
# ═══════════════════════════════════════════════════════════
from pyvis.network import Network
from pathlib import Path
from IPython.display import IFrame
import pandas as pd

OUT_DIR  = Path("kg_output")
nodes_df = pd.read_csv(OUT_DIR / "kg_nodes.csv")
edges_df = pd.read_csv(OUT_DIR / "kg_edges.csv")

# ── 取第一个 glaucoma 病例 ─────────────────────────────────
img_row  = nodes_df[
    (nodes_df.label == "FundusImage") & (nodes_df.annotation == "glaucoma")
].iloc[0]
img_id   = img_row["id"]
filename = str(img_row.get("filename", img_id)).split("/")[-1]

# ── 收集该病例直接关联的节点 id ───────────────────────────
def get_neighbors(src_id, rel):
    return set(edges_df[
        (edges_df.src == src_id) & (edges_df.rel == rel)
    ]["dst"].values)

od_id   = list(get_neighbors(img_id, "HAS_OPTIC_DISC"))[0]
rim_id  = list(get_neighbors(img_id, "HAS_RIM"))[0]
path_id = list(get_neighbors(img_id, "HAS_PATHOLOGY"))[0]
diag_id = list(get_neighbors(img_id, "HAS_DIAGNOSIS"))[0]
risk_id = list(get_neighbors(diag_id, "HAS_RISK"))[0]

# 只取该病例 Diagnosis 实际支持的规则（SUPPORTS_RULE）
rule_ids = list(get_neighbors(diag_id, "SUPPORTS_RULE"))

# 对应的 GOVERNED_BY：只保留 biomarker→rule（该病例激活的）
governed_edges = edges_df[
    (edges_df.rel == "GOVERNED_BY") &
    (edges_df.src.isin([od_id, rim_id, path_id])) &
    (edges_df.dst.isin(rule_ids))
].drop_duplicates(subset=["src", "dst"])  # 去重

all_node_ids = {img_id, od_id, rim_id, path_id, diag_id, risk_id} | set(rule_ids)
sub_nodes = nodes_df[nodes_df.id.isin(all_node_ids)]

# ── 固定坐标：三列分层布局 ────────────────────────────────
#
#  Col A (x=-500): FundusImage
#  Col B (x=-200): OpticDisc / NeuralRim / Pathology / Diagnosis / RiskLevel
#  Col C (x= 200): ClinicalRule 节点（垂直均匀排列）
#

node_lookup = sub_nodes.set_index("id")

# Col B y 坐标
COL_B = {
    od_id:   -240,
    rim_id:  -80,
    path_id:  80,
    diag_id:  240,
    risk_id:  380,
}

# Col C: 规则节点按 y 均匀分布
n_rules = len(rule_ids)
rule_ys = {rid: int(-320 + i * (640 / max(n_rules - 1, 1)))
           for i, rid in enumerate(rule_ids)}

NODE_STYLE = {
    "FundusImage":  {"color": "#7B68EE", "shape": "dot",     "size": 30},
    "OpticDisc":    {"color": "#20B2AA", "shape": "dot",     "size": 24},
    "NeuralRim":    {"color": "#3CB371", "shape": "dot",     "size": 24},
    "Pathology":    {"color": "#FF8C00", "shape": "dot",     "size": 24},
    "Diagnosis":    {"color": "#DC143C", "shape": "diamond", "size": 28},
    "RiskLevel":    {"color": "#B22222", "shape": "star",    "size": 26},
    "ClinicalRule": {"color": "#4169E1", "shape": "square",  "size": 20},
}

STRENGTH_WIDTH = {"strong": 2.2, "moderate": 1.4, "weak": 0.8}

net = Network(height="750px", width="100%", bgcolor="#1a1a2e",
              font_color="white", directed=True)
net.set_options("""
{
  "physics": {"enabled": false},
  "edges": {
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.8}},
    "smooth": {"type": "curvedCW", "roundness": 0.2},
    "font":   {"size": 11, "color": "#dddddd",
               "strokeWidth": 2, "strokeColor": "#1a1a2e"}
  },
  "nodes": {"font": {"size": 13, "face": "Arial"}},
  "interaction": {"dragNodes": true, "zoomView": true}
}
""")

# ── 添加节点 ──────────────────────────────────────────────
for _, row in sub_nodes.iterrows():
    nid   = row["id"]
    label = row["label"]
    s     = NODE_STYLE.get(label, {"color": "#888", "shape": "dot", "size": 18})

    # 坐标
    if nid == img_id:
        x, y = -500, 70
    elif nid in COL_B:
        x, y = -200, COL_B[nid]
    elif nid in rule_ys:
        x, y = 250, rule_ys[nid]
    else:
        x, y = 0, 0

    # 显示标签
    if label == "FundusImage":
        display = filename
    elif label == "ClinicalRule":
        display = str(row.get("name", nid))
    elif label == "RiskLevel":
        display = str(row.get("value", nid))
    elif label == "Diagnosis":
        display = f"Diagnosis\n{row.get('risk_assessment','')}"
    else:
        display = label

    # tooltip
    props = {k: v for k, v in row.items()
             if k not in ("id","label") and pd.notna(v) and str(v) != "nan"}
    tooltip = f"<b>{label}</b><br>" + "<br>".join(
        f"{k}: {v}" for k, v in list(props.items())[:8]
    )

    net.add_node(nid, label=display, color=s["color"], shape=s["shape"],
                 size=s["size"], x=x, y=y, physics=False,
                 title=tooltip, font={"size": 13, "color": "white"})

# ── 添加边 ────────────────────────────────────────────────
EDGE_STYLE = {
    "HAS_OPTIC_DISC":    ("#20B2AA", 2.0, False),
    "HAS_RIM":           ("#3CB371", 2.0, False),
    "HAS_PATHOLOGY":     ("#FF8C00", 2.0, False),
    "HAS_DIAGNOSIS":     ("#DC143C", 2.5, False),
    "HAS_RISK":          ("#B22222", 2.0, False),
    "SUPPORTS_DIAGNOSIS":("#888888", 1.2, True),
    "GOVERNED_BY":       ("#4169E1", 1.0, True),
    "SUPPORTS_RULE":     ("#6A5ACD", 1.2, True),
}

def add_edges(df_edges, rel, label_fn=None):
    subset = df_edges[df_edges.rel == rel].drop_duplicates(subset=["src","dst"])
    color, width, dashes = EDGE_STYLE.get(rel, ("#aaa", 1.0, False))
    for _, e in subset.iterrows():
        if e["src"] not in all_node_ids or e["dst"] not in all_node_ids:
            continue
        elabel = label_fn(e) if label_fn else rel
        net.add_edge(str(e["src"]), str(e["dst"]),
                     label=elabel, color=color, width=width, dashes=dashes,
                     font={"size": 10, "color": "#cccccc",
                           "strokeWidth": 2, "strokeColor": "#1a1a2e"})

# 结构边
for rel in ("HAS_OPTIC_DISC", "HAS_RIM", "HAS_PATHOLOGY", "HAS_DIAGNOSIS", "HAS_RISK"):
    add_edges(edges_df[edges_df.src.isin(all_node_ids)], rel)

# SUPPORTS_DIAGNOSIS（带 evidence 标签）
add_edges(edges_df[edges_df.src.isin(all_node_ids)], "SUPPORTS_DIAGNOSIS",
          label_fn=lambda e: f"SUPPORTS\n({e.get('evidence','')})")

# GOVERNED_BY（已去重）
for _, e in governed_edges.iterrows():
    color, width, dashes = EDGE_STYLE["GOVERNED_BY"]
    # 线宽按规则 strength
    rule_row = nodes_df[nodes_df.id == e["dst"]]
    strength = rule_row["strength"].values[0] if len(rule_row) and "strength" in rule_row.columns else "moderate"
    w = STRENGTH_WIDTH.get(str(strength), 1.0)
    net.add_edge(str(e["src"]), str(e["dst"]),
                 label="GOVERNED_BY", color=color, width=w, dashes=dashes,
                 font={"size": 10, "color": "#cccccc",
                       "strokeWidth": 2, "strokeColor": "#1a1a2e"})

# SUPPORTS_RULE
add_edges(edges_df[edges_df.src == diag_id], "SUPPORTS_RULE")

# ── 保存 & 修复本地依赖 ───────────────────────────────────
path = OUT_DIR / "fig2_case_v2.html"
net.save_graph(str(path))
content = path.read_text(encoding="utf-8")
content = content.replace('<script src="lib/bindings/utils.js"></script>', '')
path.write_text(content, encoding="utf-8")

print(f"Saved: {path}")
print(f"Nodes: {len(all_node_ids)}  (biomarkers: 3, rules: {n_rules})")
IFrame(str(path), width="100%", height=770)

Saved: kg_output/fig2_case_v2.html
Nodes: 14  (biomarkers: 3, rules: 8)


In [14]:
# ═══════════════════════════════════════════════════════════
# Cell — 图3 重做：聚合概览图，论文级别
# 策略：每类节点聚成群，采样少量实例展示，边只展示类型连接
# ═══════════════════════════════════════════════════════════
from pyvis.network import Network
from pathlib import Path
from IPython.display import IFrame
import pandas as pd
import numpy as np

OUT_DIR  = Path("kg_output")
nodes_df = pd.read_csv(OUT_DIR / "kg_nodes.csv")
edges_df = pd.read_csv(OUT_DIR / "kg_edges.csv")

# ── 统计信息（用于标签）────────────────────────────────────
label_counts = nodes_df["label"].value_counts().to_dict()
edge_counts  = edges_df["rel"].value_counts().to_dict()

g_count = (nodes_df[nodes_df.label=="FundusImage"]["annotation"] == "glaucoma").sum()
n_count = (nodes_df[nodes_df.label=="FundusImage"]["annotation"] == "normal").sum()

# ── 网络初始化 ─────────────────────────────────────────────
net = Network(height="780px", width="100%", bgcolor="#0f0f1e",
              font_color="white", directed=True)
net.set_options("""
{
  "physics": {"enabled": false},
  "edges": {
    "arrows": {"to": {"enabled": true, "scaleFactor": 1.0}},
    "smooth": {"type": "curvedCW", "roundness": 0.25},
    "font":   {"size": 12, "color": "#dddddd",
               "strokeWidth": 2, "strokeColor": "#0f0f1e"}
  },
  "nodes": {"font": {"size": 14, "face": "Arial", "bold": true}},
  "interaction": {"dragNodes": true, "zoomView": true}
}
""")

# ── 布局坐标（固定，论文友好）─────────────────────────────
#
#   [FundusImage glaucoma]  [FundusImage normal]
#          |                       |
#     [OpticDisc]  [NeuralRim]  [Pathology]
#               \       |       /
#              [Diagnosis]   [RiskLevel]
#                    |
#             [ClinicalRule]
#

CLUSTER_POS = {
    "FundusImage_glaucoma": (-350,  -300),
    "FundusImage_normal":   ( 350,  -300),
    "OpticDisc":            (-350,   -50),
    "NeuralRim":            (   0,   -50),
    "Pathology":            ( 350,   -50),
    "Diagnosis":            (   0,   200),
    "RiskLevel":            ( 400,   200),
    "ClinicalRule":         (   0,   430),
}

CLUSTER_STYLE = {
    "FundusImage_glaucoma": {"color": "#DC143C", "shape": "dot",     "size": 55},
    "FundusImage_normal":   {"color": "#7B68EE", "shape": "dot",     "size": 45},
    "OpticDisc":            {"color": "#20B2AA", "shape": "dot",     "size": 38},
    "NeuralRim":            {"color": "#3CB371", "shape": "dot",     "size": 38},
    "Pathology":            {"color": "#FF8C00", "shape": "dot",     "size": 38},
    "Diagnosis":            {"color": "#C41E3A", "shape": "diamond", "size": 44},
    "RiskLevel":            {"color": "#B22222", "shape": "star",    "size": 36},
    "ClinicalRule":         {"color": "#4169E1", "shape": "square",  "size": 34},
}

CLUSTER_LABELS = {
    "FundusImage_glaucoma": f"Glaucoma\nn={g_count}",
    "FundusImage_normal":   f"Normal\nn={n_count}",
    "OpticDisc":            f"OpticDisc\nn={label_counts.get('OpticDisc',0)}",
    "NeuralRim":            f"NeuralRim\nn={label_counts.get('NeuralRim',0)}",
    "Pathology":            f"Pathology\nn={label_counts.get('Pathology',0)}",
    "Diagnosis":            f"Diagnosis\nn={label_counts.get('Diagnosis',0)}",
    "RiskLevel":            f"RiskLevel\nn={label_counts.get('RiskLevel',0)}",
    "ClinicalRule":         f"ClinicalRule\nn={label_counts.get('ClinicalRule',0)}",
}

CLUSTER_TOOLTIP = {
    "FundusImage_glaucoma": f"Glaucoma fundus images<br>Count: {g_count}<br>Positive class",
    "FundusImage_normal":   f"Normal fundus images<br>Count: {n_count}<br>Negative class",
    "OpticDisc":            f"Optic disc biomarker nodes<br>CDR, disc size, sharp edge<br>Count: {label_counts.get('OpticDisc',0)}",
    "NeuralRim":            f"Neural rim biomarker nodes<br>ISNT rule, rim thinning, pallor<br>Count: {label_counts.get('NeuralRim',0)}",
    "Pathology":            f"Pathology sign nodes<br>Bayoneting, notching, LDS<br>Count: {label_counts.get('Pathology',0)}",
    "Diagnosis":            f"Diagnosis nodes<br>Risk assessment + confidence<br>Count: {label_counts.get('Diagnosis',0)}",
    "RiskLevel":            f"Risk level nodes (shared)<br>high risk / moderate / healthy<br>Count: {label_counts.get('RiskLevel',0)}",
    "ClinicalRule":         f"Clinical rule nodes<br>CDR≥0.7, ISNT rule, etc.<br>Count: {label_counts.get('ClinicalRule',0)}",
}

# ── 添加聚合节点 ──────────────────────────────────────────
for cid, (x, y) in CLUSTER_POS.items():
    s = CLUSTER_STYLE[cid]
    net.add_node(
        cid,
        label=CLUSTER_LABELS[cid],
        color=s["color"], shape=s["shape"], size=s["size"],
        x=x, y=y, physics=False,
        title=CLUSTER_TOOLTIP[cid],
        font={"size": 14, "color": "white", "bold": True},
        borderWidth=2,
    )

# ── 聚合边（带数量标签）───────────────────────────────────
CLUSTER_EDGES = [
    # 结构边（实线）
    ("FundusImage_glaucoma", "OpticDisc",  "HAS_OPTIC_DISC",
     "#20B2AA", 2.5, False, edge_counts.get("HAS_OPTIC_DISC", 0)),
    ("FundusImage_glaucoma", "NeuralRim",  "HAS_RIM",
     "#3CB371", 2.5, False, edge_counts.get("HAS_RIM", 0)),
    ("FundusImage_glaucoma", "Pathology",  "HAS_PATHOLOGY",
     "#FF8C00", 2.5, False, edge_counts.get("HAS_PATHOLOGY", 0)),
    ("FundusImage_glaucoma", "Diagnosis",  "HAS_DIAGNOSIS",
     "#DC143C", 3.0, False, edge_counts.get("HAS_DIAGNOSIS", 0)),
    ("FundusImage_normal",   "OpticDisc",  "HAS_OPTIC_DISC",
     "#20B2AA", 2.0, False, ""),
    ("FundusImage_normal",   "NeuralRim",  "HAS_RIM",
     "#3CB371", 2.0, False, ""),
    ("FundusImage_normal",   "Pathology",  "HAS_PATHOLOGY",
     "#FF8C00", 2.0, False, ""),
    ("FundusImage_normal",   "Diagnosis",  "HAS_DIAGNOSIS",
     "#9370DB", 2.5, False, ""),
    ("Diagnosis", "RiskLevel", "HAS_RISK",
     "#B22222", 2.0, False, edge_counts.get("HAS_RISK", 0)),
    # 推理边（虚线）
    ("OpticDisc",  "Diagnosis", "SUPPORTS_DIAGNOSIS",
     "#666666", 1.5, True, edge_counts.get("SUPPORTS_DIAGNOSIS", 0)),
    ("NeuralRim",  "Diagnosis", "SUPPORTS_DIAGNOSIS",
     "#666666", 1.5, True, ""),
    ("Pathology",  "Diagnosis", "SUPPORTS_DIAGNOSIS",
     "#666666", 1.5, True, ""),
    # 规则边（虚线）
    ("OpticDisc",  "ClinicalRule", "GOVERNED_BY",
     "#4169E1", 1.2, True, ""),
    ("NeuralRim",  "ClinicalRule", "GOVERNED_BY",
     "#4169E1", 1.2, True, ""),
    ("Pathology",  "ClinicalRule", "GOVERNED_BY",
     "#4169E1", 1.2, True, ""),
    ("Diagnosis",  "ClinicalRule", "SUPPORTS_RULE",
     "#6A5ACD", 1.5, True, edge_counts.get("SUPPORTS_RULE", 0)),
]

for src, dst, rel, color, width, dashes, count in CLUSTER_EDGES:
    elabel = f"{rel}\n({count})" if count else rel
    net.add_edge(src, dst,
                 label=elabel, color=color, width=width, dashes=dashes,
                 font={"size": 10, "color": "#cccccc",
                       "strokeWidth": 2, "strokeColor": "#0f0f1e"})

# ── 图例（右下角用独立节点模拟）──────────────────────────
legend_items = [
    ("leg_solid",  "─── Structural edge", "#aaaaaa", "dot",     8, 520, 320),
    ("leg_dash",   "- - Inference edge",  "#666666", "dot",     8, 520, 360),
    ("leg_rule",   "- - Rule edge",       "#4169E1", "dot",     8, 520, 400),
]
for lid, llabel, lcolor, lshape, lsize, lx, ly in legend_items:
    net.add_node(lid, label=llabel, color=lcolor, shape=lshape,
                 size=lsize, x=lx, y=ly, physics=False,
                 font={"size": 11, "color": "#aaaaaa"})

# ── 保存 & 修复 ───────────────────────────────────────────
path = OUT_DIR / "fig3_overview_v2.html"
net.save_graph(str(path))
content = path.read_text(encoding="utf-8")
content = content.replace('<script src="lib/bindings/utils.js"></script>', '')
path.write_text(content, encoding="utf-8")

print(f"Saved: {path}")
print(f"Cluster nodes: {len(CLUSTER_POS)}")
print(f"Cluster edges: {len(CLUSTER_EDGES)}")
IFrame(str(path), width="100%", height=800)

Saved: kg_output/fig3_overview_v2.html
Cluster nodes: 8
Cluster edges: 16
